# Milestone 3 lab, notebook 4 of 8: At least once, exactly once

## Where we are

| Notebook | What you do |
|---|---|
| 1. One order, one row | Write the same rows three ways: append, overwrite, upsert |
| 2. Which column is the key? | Merge on the wrong key. Then meet an order that arrives twice in one run |
| 3. Run it twice | Prove that a second run changes nothing |
| **4. At least once, exactly once** (you are here) | Crash the load in three places, and see what each one costs |
| 5. The update that came late | Find a change the watermark has passed. Catch it with a lookback |
| 6. Fix one day | Break Friday's numbers, and rebuild Friday alone |
| 7. The order that vanished | A deleted order, and why the query cannot see it |
| 8. The production version | The three properties, as tests |

A load does two things: it **writes rows**, and it **saves how far it got**. A crash can happen between them. Which one you do first decides what a crash costs. This notebook crashes the same load in three places.

In [1]:
import pandas as pd

from lab_helpers import lab_warehouse, read_orders, run_pipeline, set_morning, show_doc, table, true_day_totals

## Step 1: The set-up

Monday's 226 orders are in the warehouse, and the watermark is saved. Now Tuesday's load runs, crashes once, and is run again. The right result is 251 orders.

In [2]:
set_morning("2026-06-08")
monday, _ = read_orders()
monday_watermark = max(order["updated_at"] for order in monday)

set_morning("2026-06-09")
print("After Monday:", len(monday), "orders. Watermark:", monday_watermark)

It is now Monday 2026-06-08 07:00:00 in Arden.
It is now Tuesday 2026-06-09 07:00:00 in Arden.
After Monday: 226 orders. Watermark: 2026-06-08T03:48:53Z


Each try below is a tiny pipeline in plain Python: a list or a dictionary for the table, and a variable for the watermark.

## Way A: save the watermark first

In [3]:
def run_tuesday(table_rows, watermark, write, crash_after_first_step=False, watermark_first=False):
    """One run of Tuesday's load. Returns the new watermark."""
    changed, _ = read_orders(updated_since=watermark)
    newest = max(order["updated_at"] for order in changed)
    if watermark_first:
        watermark = newest
        if crash_after_first_step:
            return watermark                      # 💥 the rows were never written
        write(table_rows, changed)
    else:
        write(table_rows, changed)
        if crash_after_first_step:
            return watermark                      # 💥 the watermark was never saved
        watermark = newest
    return watermark

def append(rows, changed):
    rows.extend(changed)

rows = list(monday)
watermark = run_tuesday(rows, monday_watermark, append, crash_after_first_step=True, watermark_first=True)   # crash
watermark = run_tuesday(rows, watermark, append, watermark_first=True)                                        # the retry

way_a = {"rows in table": len(rows), "different orders": len({o["order_id"] for o in rows})}
way_a

{'rows in table': 227, 'different orders': 227}

The retry started from the **new** watermark, so it asked only for the boundary order. Tuesday's orders were never written: 24 orders are missing, and no error was shown.

A load that can lose rows but never repeats them is called **at-most-once**.

## Way B: write first, with an append

In [4]:
rows = list(monday)
watermark = run_tuesday(rows, monday_watermark, append, crash_after_first_step=True)    # crash
watermark = run_tuesday(rows, watermark, append)                                         # the retry

way_b = {"rows in table": len(rows), "different orders": len({o["order_id"] for o in rows})}
way_b

{'rows in table': 284, 'different orders': 251}

Nothing is lost: all 251 orders are there. But the retry asked from the **old** watermark, got Tuesday's rows a second time, and appended them again.

A load that never loses a row but may repeat one is called **at-least-once**.

## Way C: write first, by key

In [5]:
def write_by_key(orders_by_id, changed):
    for order in changed:
        orders_by_id[order["order_id"]] = order          # an upsert, in one line of Python

orders_by_id = {order["order_id"]: order for order in monday}
watermark = run_tuesday(orders_by_id, monday_watermark, write_by_key, crash_after_first_step=True)   # crash
watermark = run_tuesday(orders_by_id, watermark, write_by_key)                                        # the retry

way_c = {"rows in table": len(orders_by_id), "different orders": len(orders_by_id)}
way_c

{'rows in table': 251, 'different orders': 251}

The rows arrived twice, exactly as in way B. The delivery is still at-least-once. But the write is by key, so the second arrival changed nothing.

In [6]:
pd.DataFrame([
    {"way": "A: watermark first", **way_a, "result": "❌ orders lost", "promise": "at-most-once"},
    {"way": "B: rows first, append", **way_b, "result": "❌ orders doubled", "promise": "at-least-once"},
    {"way": "C: rows first, by key", **way_c, "result": "✅ right", "promise": "at-least-once delivery, exactly-once result"},
])

,way,rows in table,different orders,result,promise
0,A: watermark first,227,227,❌ orders lost,at-most-once
1,"B: rows first, append",284,251,❌ orders doubled,at-least-once
2,"C: rows first, by key",251,251,✅ right,"at-least-once delivery, exactly-once result"


## Step 2: Three promises

| Promise | It means | Can lose a row? | Can repeat a row? |
|---|---|---|---|
| **At-most-once** | Never send again | Yes | No |
| **At-least-once** | Send again until it worked | No | Yes |
| **Exactly-once** | Each row changes the result one time | No | No |

For money, a lost order is worse than a repeated order. So you choose at-least-once, and you make repeats harmless.

Look again at way C. Nothing in the network became "exactly once". The same rows still travelled twice. What changed is the write:

> "Exactly-once" usually means at-least-once delivery plus an idempotent write.

## Step 3: The real pipeline, on the morning with the repeated page

On Thursday 11 June the order system serves one page twice. That is a repeat you did not cause and cannot stop. Run the production pipeline through that morning.

In [7]:
con = lab_warehouse(fresh=True)
result = run_pipeline(con, "2026-06-08", "2026-06-09", "2026-06-10", "2026-06-11")

table(con, """
    SELECT count(*) AS rows_landed_on_thursday,
           count(DISTINCT payload->>'order_id') AS different_orders,
           (SELECT count(*) FROM core.orders) AS orders_in_clean_table
    FROM raw.orders_landing WHERE run_id = '2026-06-11_1'
""")

,rows_landed_on_thursday,different_orders,orders_in_clean_table
0,120,95,317


In [8]:
from daily_orders.report import daily_report, day_totals

report, truth = day_totals(daily_report(con)), true_day_totals()
same = all((report[day]["orders"], report[day]["revenue_cents"]) == (t["orders"], t["revenue_cents"]) for day, t in truth.items())
print("The report equals the truth on every day:", same)
con.close()

The report equals the truth on every day: True


More rows landed than there are orders, and the report is still right.

In [9]:
show_doc("HLD.md", "3. The idea that makes it safe")

**From `docs/HLD.md`**

> ## 3. The idea that makes it safe
> 
> Milestone 2 was safe from **loss**, and paid for it with **repeats**. This milestone does not stop the repeats. It makes the write not care.
> 
> *(There is a diagram here. Open `docs/HLD.md` to see it.)*
> 
> | Promise | It means | Can lose a row? | Can repeat a row? |
> |---|---|---|---|
> | At-most-once | Never send again | Yes | No |
> | **At-least-once** (our delivery) | Send again until it worked | No | Yes |
> | **Exactly-once** (our result) | Each row changes the result one time | No | No |
> 
> Exactly-once is not something a network can give you. It is at-least-once delivery **plus** an idempotent write.

## New words

| Word | Meaning | In this notebook |
|---|---|---|
| **At-most-once** | A row is never sent twice, but it can be lost | Way A: 24 orders lost |
| **At-least-once** | A row is never lost, but it can arrive twice | Ways B and C: Tuesday's rows arrived twice |
| **Exactly-once** | Each row changes the result one time | Way C: 251 orders |

The definitions are from the D8Loop chapters [Batch vs Streaming vs Micro-Batch](https://www.d8loop.com/modules/knowledge-corner/data-pipeline/batch-vs-streaming-vs-micro-batch) and [Idempotency, Retries & Exactly-Once](https://www.d8loop.com/modules/knowledge-corner/data-pipeline/idempotency-retries-and-exactly-once).

## What you now know

- Save the watermark first, and a crash loses rows: at-most-once.
- Write the rows first, and a crash repeats them: at-least-once.
- Write by key, and the repeat does no harm: an exactly-once result.
- Exactly-once is a property of the source, the job and the table together. No single part can promise it.

**Say it in an interview:** "An API call can only be at-least-once, because a retry may bring a page again. I do not try to prevent that. I write first and save the watermark after, in one transaction, and the write is a MERGE on the order's key. So delivery is at-least-once and the result in the table is exactly-once."

Next: [notebook 5](05_the_update_that_came_late.ipynb) finds a change that this pipeline would still miss without one more idea.